In [ ]:
%pip install 'stable-baselines3[extra]'
%pip install sb3-contrib

In [7]:
import os
import time
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import BaseCallback

from blackjack_env.wrappers import ShiftWrapper, SafeV1ActionWrapper
from blackjack_env.agent_utils.sb3_ppo import test_env_PPO

log_dir = os.path.abspath(f"../results/sb3_blackjack_{int(time.time())}")
os.makedirs(log_dir, exist_ok=True)


def make_env(log_dir: str, seed: int) -> gym.Env:
    env = gym.make("Blackjack4game-v1", render_mode=None)
    env = ShiftWrapper(env)
    env = SafeV1ActionWrapper(env)
    env = Monitor(env, filename=os.path.join(log_dir, "monitor.csv"))
    env.reset(seed=seed)
    return env


def make_evaluation_env(seed: int) -> gym.Env:
    env = gym.make("Blackjack4game-v1", render_mode=None)
    env = ShiftWrapper(env)
    env = SafeV1ActionWrapper(env)
    env.reset(seed=seed)
    return env

In [ ]:
ppo_kwargs_base = dict(
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
)

ppo_kwargs_stable_slow = dict(
    learning_rate=1e-4,
    n_steps=4096,
    batch_size=256,
    n_epochs=10,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.15,
    ent_coef=0.005,
)

ppo_kwargs_fast = dict(
    learning_rate=1e-3,
    n_steps=1024,
    batch_size=128,
    n_epochs=5,
    gamma=0.99,
    gae_lambda=0.9,
    clip_range=0.2,
    ent_coef=0.02,
)

ppo_kwargs_explore = dict(
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    ent_coef=0.05,  # większa entropia
    gamma=0.99,
    gae_lambda=0.95,
)

In [ ]:
test_env = make_evaluation_env(seed=42)
eval_env = make_evaluation_env(seed=42)

test_env_PPO(
    env_train=test_env,
    env_eval=eval_env,
    policy="MultiInputPolicy",
    verbose=1,
    tensorboard_log_dir=log_dir,
    n_runs=10,
    run_timesteps=50_000,
    eval_freq=10_000,
    n_eval_episodes=1000,
    base_seed=42,
    ppo_kwargs=ppo_kwargs_base,
)